# Expanded Heart Disease Data Analysis - SKELETON (R Version)

## Practice Notebook in R (Fill in the Code)

**Objective:** Practice statistical analysis and data visualization in **R** on the heart disease dataset, expanding the original exercises with audience-aware communication, alternates, simulation, and professional reporting structure.

**Key Additions (same as Python version):**
- More predictors (`sex`, `exang`, `fbs`)
- Effect sizes + non-parametric alternates (wilcox.test, etc.)
- Logistic regression (`glm` family=binomial)
- Simulation & sensitivity (bootstrap, permutation via `replicate` or `boot` package)
- Flowchart of workflow + audience considerations from the attached PDFs
- Structured data analysis report outline

**How to use:**
1. Install IRkernel if needed: `install.packages("IRkernel"); IRkernel::installspec()`
2. Open this notebook in Jupyter with R kernel
3. Fill `# YOUR CODE HERE` cells
4. Compare with the **solution_R notebook** for full code, alternates, and printed outputs

**Dataset path:** `/home/workdir/attachments/hearth_disease.csv`

**Recommended libraries (install once):**
`tidyverse`, `ggplot2`, `broom`, `boot` (for bootstrap)

## 1. Setup and Library Imports

In [ ]:
# Install if needed (run once)
# install.packages(c("tidyverse", "ggplot2", "broom", "boot"))

library(tidyverse)
library(ggplot2)
library(broom)   # for tidy model output
library(boot)    # optional for bootstrap

# Colorblind-friendly theme
theme_set(theme_minimal() + 
          theme(legend.position = "bottom",
                plot.title = element_text(face = "bold", size = 12)))

cat("Libraries loaded. Ready for R analysis!
")

## 2. Flowchart of Desired Analysis Outcome (R Version)

**Study this workflow** — it is the professional standard incorporating audience analysis (data literacy, subject knowledge) and structured reporting.

The same flowchart image is embedded below (generated previously). It shows the 7-step process you should follow for rigorous, audience-aware analysis.

![Analysis Workflow Flowchart](/home/workdir/artifacts/analysis_flowchart.png)

## 3. Data Loading and Initial Inspection

**Task:** Load the CSV, inspect with `head()`, `glimpse()`, `summary()`, check missing values and factor levels for categorical variables.

**Audience note:** For low data-literacy readers, start with simple counts and plain-language variable explanations. For clinicians/data scientists, show distributions and missingness immediately.

In [ ]:
# YOUR CODE HERE
heart <- read_csv("/home/workdir/attachments/hearth_disease.csv")

cat("=== First 6 rows ===
")
print(head(heart))

cat("
=== Glimpse (structure) ===
")
glimpse(heart)

cat("
=== Summary statistics ===
")
summary(heart)

cat("
=== Missing values ===
")
print(colSums(is.na(heart)))

cat("
=== Categorical value counts ===
")
for (col in c("sex", "cp", "heart_disease", "exang", "fbs")) {
  cat("
", col, ":
")
  print(table(heart[[col]]))
}

## 4. EDA Visualizations (Audience-Adapted) - R / ggplot2

**Task 4.1:** Boxplot of `thalach` by `heart_disease` with mean annotations and clear title.

**Task 4.2:** Countplot (bar) of `cp` by `heart_disease` — keep simple for mixed audiences.

**Task 4.3 (technical audiences only):** Correlation heatmap of quantitative variables using `ggcorrplot` or base `corrplot` (install if needed).

In [ ]:
# YOUR CODE HERE - Task 4.1
ggplot(heart, aes(x = heart_disease, y = thalach, fill = heart_disease)) +
  geom_boxplot() +
  stat_summary(fun = mean, geom = "point", shape = 23, size = 3, fill = "white") +
  labs(title = "Maximum Heart Rate (thalach) by Heart Disease Status",
       subtitle = "Higher thalach generally associated with absence of disease",
       x = "Heart Disease", y = "Max Heart Rate (bpm)") +
  theme(legend.position = "none")

# Task 4.2 - simple bar for cp
ggplot(heart, aes(x = cp, fill = heart_disease)) +
  geom_bar(position = "dodge") +
  labs(title = "Chest Pain Type Distribution by Heart Disease",
       x = "Chest Pain Type", y = "Count") +
  theme(axis.text.x = element_text(angle = 20, hjust = 1))

# Task 4.3 (advanced - uncomment after installing ggcorrplot)
# library(ggcorrplot)
# quant_vars <- c("age", "trestbps", "chol", "thalach")
# corr <- cor(heart[quant_vars], use = "complete.obs")
# ggcorrplot(corr, lab = TRUE, title = "Correlation Heatmap (Technical Audiences Only)")

## 5. Univariate Hypothesis Tests (R)

**Task 5.1 - thalach:**
- Split into two vectors
- Mean & median differences
- `t.test` (primary)
- **Alternate:** `wilcox.test` (Mann-Whitney)
- Effect size (Cohen's d or rank-biserial)

Repeat for `age`, `trestbps`, `chol` (use `par(mfrow=c(1,1))` or new plots).

**Task 5.2 - cp vs thalach:** `aov` + `TukeyHSD`

**Task 5.3 - Categorical vs heart_disease:** `chisq.test` + `fisher.test` for 2x2 tables

In [ ]:
# YOUR CODE HERE - thalach example
thalach_hd <- heart$thalach[heart$heart_disease == "presence"]
thalach_no <- heart$thalach[heart$heart_disease == "absence"]

mean_diff <- mean(thalach_no) - mean(thalach_hd)
med_diff  <- median(thalach_no) - median(thalach_hd)
cat("Mean difference (absence - presence):", round(mean_diff, 2), "
")
cat("Median difference:", round(med_diff, 2), "
")

# Primary t-test
tt <- t.test(thalach_hd, thalach_no)
cat("t-test p-value:", signif(tt$p.value, 4), "
")

# ALTERNATE: Wilcoxon / Mann-Whitney
wt <- wilcox.test(thalach_hd, thalach_no)
cat("Wilcoxon (Mann-Whitney) p-value:", signif(wt$p.value, 4), "
")

# Simple Cohen's d
pooled_sd <- sqrt( ((length(thalach_hd)-1)*var(thalach_hd) + 
                    (length(thalach_no)-1)*var(thalach_no)) / 
                   (length(thalach_hd) + length(thalach_no) - 2) )
cohens_d <- mean_diff / pooled_sd
cat("Cohen's d effect size:", round(cohens_d, 3), "
")

# Repeat the pattern for age, trestbps, chol ...

## 6. Multiple Testing Correction

Collect key p-values and apply `p.adjust(..., method = "bonferroni")` or "fdr".

In [ ]:
# YOUR CODE HERE
pvals <- c(1.7e-14, 8.2e-5, 0.011, 0.14, 1.3e-9, 1.9e-6, 2.5e-14, 0.15, 1.3e-17)
names(pvals) <- c("thalach_t", "age_t", "trestbps_t", "chol_t", "cp_anova",
                  "sex_chi2", "exang_chi2", "fbs_chi2", "cp_hd_chi2")

p_adj <- p.adjust(pvals, method = "bonferroni")
cat("Bonferroni corrected p-values:
")
print(round(p_adj, 6))
cat("
Significant after correction (alpha=0.05):", sum(p_adj < 0.05), "
")

## 7. Logistic Regression in R (`glm`)

Fit `glm(heart_disease ~ ..., family = binomial)` and interpret odds ratios with `broom::tidy` + `exp()`.

**Audience note:** Translate ORs into plain language for non-specialists and keep statistical detail for clinicians.

In [ ]:
# YOUR CODE HERE
heart <- heart %>%
  mutate(hd_binary = if_else(heart_disease == "presence", 1, 0))

model <- glm(hd_binary ~ age + thalach + chol + trestbps + 
             factor(cp) + factor(sex) + exang + fbs,
             data = heart, family = binomial)

cat("=== Model Summary (use broom::tidy for clean table) ===
")
print(summary(model))

cat("
=== Odds Ratios (exp(coef)) ===
")
or_table <- tidy(model, conf.int = TRUE, exponentiate = TRUE) %>%
  select(term, estimate, conf.low, conf.high, p.value) %>%
  arrange(desc(estimate))
print(or_table)

## 8. Simulation & Sensitivity Analysis (Modify Parameters)

**Modify at top of cell:**
- `ALPHA`
- `N_BOOT`
- `SUBSAMPLE_SIZE`

Then run bootstrap CI (using `boot` package or simple `replicate`), permutation test via `replicate`, and subsample check.

In [ ]:
# === MODIFY THESE AND RE-RUN ===
ALPHA <- 0.05
N_BOOT <- 3000
SUBSAMPLE_SIZE <- NULL   # e.g. 120
set.seed(42)

cat("Running R simulations with ALPHA =", ALPHA, "N_BOOT =", N_BOOT, "
")

# Simple bootstrap CI for mean difference (thalach)
mean_diff_fun <- function(data, indices) {
  d <- data[indices, ]
  mean(d$thalach[d$heart_disease == "absence"]) - 
  mean(d$thalach[d$heart_disease == "presence"])
}

if (requireNamespace("boot", quietly = TRUE)) {
  boot_res <- boot(heart, mean_diff_fun, R = N_BOOT)
  ci <- boot.ci(boot_res, type = "perc", conf = 1 - ALPHA)
  cat("Bootstrap", (1-ALPHA)*100, "% CI for mean thalach diff:", 
      round(ci$percent[4:5], 2), "
")
} else {
  cat("boot package not loaded - using replicate for simple bootstrap
")
  boot_diffs <- replicate(N_BOOT, {
    idx <- sample(nrow(heart), replace = TRUE)
    mean_diff_fun(heart[idx, ], 1:nrow(heart))
  })
  ci_low <- quantile(boot_diffs, ALPHA/2)
  ci_high <- quantile(boot_diffs, 1 - ALPHA/2)
  cat("Simple replicate bootstrap CI:", round(c(ci_low, ci_high), 2), "
")
}

# Permutation test (simple version)
perm_diffs <- replicate(2000, {
  shuffled <- sample(heart$heart_disease)
  mean(heart$thalach[shuffled == "absence"]) - 
  mean(heart$thalach[shuffled == "presence"])
})
obs_diff <- mean(thalach_no) - mean(thalach_hd)
p_perm <- mean(abs(perm_diffs) >= abs(obs_diff))
cat("Permutation test p-value (approx):", round(p_perm, 6), "
")

# Subsample sensitivity
if (!is.null(SUBSAMPLE_SIZE)) {
  heart_sub <- heart %>% sample_n(SUBSAMPLE_SIZE)
  tt_sub <- t.test(thalach ~ heart_disease, data = heart_sub)
  cat("Subsample (n=", SUBSAMPLE_SIZE, ") t-test p-value:", 
      signif(tt_sub$p.value, 4), "
")
}

## 9. More Practice Tasks (R)

1. Analyze `exang` with appropriate test + ggplot bar.
2. Stratified boxplot of `thalach` by `heart_disease` and `sex`.
3. Create a clean summary table of significant predictors (effect size + corrected p).
4. Write two short summaries: one for a cardiologist, one for hospital admin/patient.

Put your R code below.

In [ ]:
# YOUR CODE HERE - Practice tasks
# 1. exang
# ggplot(heart, aes(x = factor(exang), fill = heart_disease)) + geom_bar(position="fill") + ...

# 2. Stratified
# ggplot(heart, aes(x = heart_disease, y = thalach, fill = sex)) + geom_boxplot() + ...

# 4. Two summaries (text)
# specialist <- "After multivariable adjustment, each 1 bpm increase in thalach..."
# general <- "Patients who reached higher heart rates during exercise were much less likely..." 

## 10. Conclusion & Report Structure (R Version)

Summarize findings and use the professional structure:

**1. Introduction** – questions + data summary  
**2. Body** – methods brief, key visuals/tables with signposts for different audiences  
**3. Conclusion** – headlines + implications  
**4. Appendix** – full stats, code, sensitivity checks

When done, compare your R code and interpretations with the solution_R notebook.